In [2]:
%cd /home/govind/gov_semproject/GridCellsCopy

/home/govind/gov_semproject/GridCellsCopy


In [3]:
import numpy as np
import matplotlib.pyplot as plt
import g_utils
import analysis_utils as a_utils
from interneuron import Interneuron
import sim_utils as s_utils
from neuron import h

h.load_file("stdrun.hoc")
h.cvode_active(1)

1.0

In [ ]:
test_interneuron = Interneuron(0)

def f_i_const_curr(curr_amp,time_ms,sigma=50):
    
    ic = h.IClamp(test_interneuron.soma(0.5))
    ic.dur = 1e9
    T,num_steps = time_ms/1000, time_ms
    time_arr = np.linspace(0,T,num_steps)
    curr_arr = np.full_like(num_steps,curr_amp)

    curr_vec = h.Vector(curr_arr)

    curr_vec.play(ic, ic._ref_amp, True)
    rec_i = h.Vector().record(ic._ref_amp)
    rec_v = h.Vector().record(test_interneuron.soma(0.5)._ref_v)
    rec_t = h.Vector().record(h._ref_t)
    spike_times = h.Vector()

    nc = h.NetCon(test_interneuron.soma(0.5)._ref_v, None, sec=test_interneuron.soma)
    nc.threshold=1
    nc.record(spike_times)

    h.finitialize(-70)
    h.continuerun(time_ms)

    np_spike_array = np.array(spike_times.to_python())

    firing_rate = a_utils.instant_rate(spike_train=spike_times,sim_dur=time_ms,stdev=sigma)*1000 #converting to Hz
    
    del nc,ic,spike_times,rec_i,rec_v,rec_t

    return np.mean(firing_rate), np.sqrt(np.var(firing_rate))

    

curr_amp_list = [i*1e-3 for i in range(0,20)]

freq_mean_list=[]
freq_std_list =[]
for amp in curr_amp_list:
    print(f"Current : {amp}nA")
    mean,std = f_i_const_curr(curr_amp=amp,time_ms=20000,sigma=50)
    freq_mean_list.append(mean)
    freq_std_list.append(std)

plt.plot(curr_amp_list,freq_mean_list)
plt.yerr(curr_amp_list,freq_mean_list,freq_std_list)
plt.show()
     




: 